# 🔮 Customer Churn Prediction Model
## Option C — Advanced ML Pipeline
**Tools:** Logistic Regression | Random Forest | XGBoost | LightGBM | SHAP | Dash

---
### Workflow
```
Synthetic Data → EDA → Preprocessing → Feature Engineering
→ SMOTE → Model Training → Evaluation → SHAP → Business Insights
```

## 1️⃣ Setup & Imports

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

BASE = Path('..')
print('✅ Imports OK')

## 2️⃣ Generate Synthetic Dataset

In [ ]:
from src.data_generator import generate_churn_dataset

df = generate_churn_dataset(n_samples=10000, random_state=42)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Churn Distribution ===')
print(df['churn'].value_counts())
print(f'\nChurn Rate: {df["churn"].mean()*100:.2f}%')

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Churn pie
counts = df['churn'].value_counts()
axes[0].pie(counts, labels=['Retained', 'Churned'], colors=['#2ECC71','#E74C3C'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Churn Distribution', fontsize=13, fontweight='bold')

# Tenure distribution
for label, color in [(0, '#2ECC71'), (1, '#E74C3C')]:
    axes[1].hist(df[df['churn']==label]['tenure'], bins=30, alpha=0.6,
                 color=color, label='Retained' if label==0 else 'Churned', edgecolor='white')
axes[1].set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by contract type
plt.figure(figsize=(10, 5))
churn_by_contract = df.groupby('contract_type')['churn'].mean() * 100
bars = plt.bar(churn_by_contract.index, churn_by_contract.values,
               color=['#E74C3C', '#F39C12', '#2ECC71'], edgecolor='white', width=0.5)
for bar, val in zip(bars, churn_by_contract.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')
plt.title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
plt.ylabel('Churn Rate (%)')
plt.axhline(df['churn'].mean()*100, color='navy', linestyle='--', label='Overall avg')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
num_df = df.select_dtypes(include=[np.number])
corr   = num_df.corr()
mask   = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size':8})
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Preprocessing & Feature Engineering

In [ ]:
from src.preprocessing import run_full_pipeline

df.to_csv(BASE/'data'/'customer_churn_raw.csv', index=False)

X_train, X_test, y_train, y_test, encoders, scaler = run_full_pipeline(
    raw_path   = str(BASE/'data'/'customer_churn_raw.csv'),
    output_dir = str(BASE/'data')
)
print(f'Train shape: {X_train.shape}')
print(f'Test shape : {X_test.shape}')
print(f'Features   : {list(X_train.columns)}')

## 5️⃣ Handle Class Imbalance — SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

print(f'Before SMOTE — Churn: {y_train.sum()} | No Churn: {(y_train==0).sum()}')

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

print(f'After  SMOTE — Churn: {y_train_bal.sum()} | No Churn: {(y_train_bal==0).sum()}')

## 6️⃣ Model Training

In [ ]:
from src.model_training import run_training_pipeline

trained_models, results, best_name, X_test_df, y_test_s = run_training_pipeline(
    data_dir   = str(BASE/'data'),
    models_dir = str(BASE/'models')
)
print(f'\n✅ Best Model: {best_name}')

## 7️⃣ Model Evaluation

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test_s, res['y_prob'])
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.4f})')

plt.plot([0,1],[0,1],'k--',lw=1,label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Leaderboard table
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model'    : name,
        'Accuracy' : f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall'   : f"{res['recall']:.4f}",
        'F1 Score' : f"{res['f1_score']:.4f}",
        'ROC-AUC'  : f"{res['roc_auc']:.4f}",
    })
leaderboard = pd.DataFrame(summary_rows).sort_values('ROC-AUC', ascending=False)
leaderboard.index = range(1, len(leaderboard)+1)
leaderboard

## 8️⃣ SHAP Explainability

In [ ]:
import shap
from src.explainability import compute_shap_values

best_model  = trained_models[best_name]
X_sample    = X_test_df.iloc[:300].reset_index(drop=True)

shap_values, explainer, expected_val = compute_shap_values(best_model, X_sample, best_name)

# Summary Plot
plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='dot', show=True, max_display=15)

In [ ]:
# Bar plot of mean |SHAP|
plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=True, max_display=15)

## 9️⃣ Live Prediction Demo

In [ ]:
from src.predictor import ChurnPredictor, SAMPLE_CUSTOMERS, print_prediction_report

predictor    = ChurnPredictor(str(BASE/'models'))
demo_df      = pd.DataFrame(SAMPLE_CUSTOMERS)
demo_results = predictor.predict(demo_df)
print_prediction_report(demo_results)
demo_results[['customer_id','monthly_charges','tenure','contract_type',
              'churn_probability','churn_prediction','risk_level']]

## 🔟 Business Intelligence

In [ ]:
from src.business_insights import calculate_business_metrics, print_business_report

all_preds   = predictor.predict(df.drop(columns=['churn'], errors='ignore'))
biz_metrics = calculate_business_metrics(all_preds)
print_business_report(biz_metrics)

In [ ]:
print('\n🚀 All steps complete! Run the dashboard with:')
print('   python src/dashboard.py')
print('   Then open: http://127.0.0.1:8050')